# Notebook 16 — Capstone: end-to-end credit portfolio workflow

**Prerequisites:** complete the full finstack curriculum **notebooks 01–15** before this capstone. You should be fluent with core types and money, dates and schedules, market data and curves, the math toolkit, instrument pricing, portfolio construction, financial statement modeling, performance analytics, risk and factor analytics, and scenarios / stress testing.

This notebook ties those pieces into one linear story: **market snapshot → price instruments → aggregate a portfolio → model an issuer → stress the market → measure performance and risk → explain model outputs for reporting**.


## Concept: an integrated credit portfolio workflow

Real credit and multi-asset desks rarely use a single API in isolation. A typical internal workflow:

1. **Market setup** — Build a `MarketContext` (discount and forward curves, FX) as the valuation snapshot for a single **as-of** date.
2. **Instrument pricing** — Serialize instruments as JSON, call `price_instrument_with_metrics`, and wrap results in `ValuationResult` for PV and metrics.
3. **Portfolio construction** — Describe entities and positions in portfolio JSON, call `value_portfolio`, then read **`total_base_ccy`** from the valuation JSON or wrap a full **`PortfolioResult`** for `portfolio_result_total_value`.
4. **Issuer modeling** — Use `ModelBuilder` / `Evaluator` for a simple P&L bridge (revenue → EBITDA → pre-tax) aligned with credit surveillance.
5. **Stress testing** — Inspect built-in **templates** (`list_builtin_templates`, `build_from_template`), then apply a **small custom scenario** (`build_scenario_spec` + `apply_scenario_to_market`) that only references curves present in this notebook.
6. **Performance analysis** — On a synthetic price path, use `Performance` for CAGR, Sharpe, drawdowns, and standalone risk statistics.
7. **Risk analytics** — Estimate beta vs a benchmark return series and **ruin probability** under a wealth-floor definition.
8. **Reporting** — Use `statements_analytics` to trace dependencies and explain formulas for a line item and period.

The code below is **self-contained** (no external data files). Run cells **top to bottom**.


## Step 1 — Market setup

Construct a **`MarketContext`**: USD OIS discount curve, USD SOFR forward curve, and a spot **EUR/USD** quote via `FxMatrix`. Serialize with **`to_json()`** so downstream pricing and portfolio calls share one market snapshot string.


In [1]:
from datetime import date

from finstack.core.currency import Currency
from finstack.core.market_data import DiscountCurve, ForwardCurve, FxMatrix, MarketContext

mc = MarketContext()
# USD OIS discount curve
ois = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [(0.0, 1.0), (0.25, 0.9875), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    day_count="act_365f",
)
mc.insert(ois)
# USD SOFR forward curve
sofr = ForwardCurve(
    "USD-SOFR",
    0.25,
    [(0.0, 0.045), (1.0, 0.048), (2.0, 0.05), (5.0, 0.052)],
    base_date=date(2025, 1, 15),
    day_count="act_360",
)
mc.insert(sofr)
# FX
fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.08)
mc.insert_fx(fx)
market_json = mc.to_json()
print(f"MarketContext JSON length: {len(market_json):,} characters")
print("Curves and FX inserted for as-of 2025-01-15.")


MarketContext JSON length: 1,518 characters
Curves and FX inserted for as-of 2025-01-15.


## Step 2 — Instrument pricing

Encode a **USD deposit** as JSON (type + `spec`), price with **`price_instrument_with_metrics`**, and parse the result with **`ValuationResult.from_json`**. The present value is exposed as the **`price`** property alongside **`currency`**.


In [2]:
import json

from finstack.valuations import ValuationResult, price_instrument_with_metrics

deposit = json.dumps(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-6M",
            "notional": {"amount": 5000000.0, "currency": "USD"},
            "start_date": "2025-01-15",
            "maturity": "2025-07-15",
            "day_count": "Act360",
            "quote_rate": 0.048,
            "discount_curve_id": "USD-OIS",
            "attributes": {"meta": {"sector": "Cash"}, "tags": ["cash"]},
        },
    }
)
dep_result_json = price_instrument_with_metrics(deposit, market_json, "2025-01-15")
dep_vr = ValuationResult.from_json(dep_result_json)
print(f"Deposit PV: {dep_vr.price:,.2f} {dep_vr.currency}")
print(f"Instrument id: {dep_vr.instrument_id}")


Deposit PV: -6,297.70 USD
Instrument id: DEP-6M


## Step 3 — Portfolio construction

Define a **portfolio spec** JSON with entities and positions. The deposit position embeds the same instrument dict via **`json.loads(deposit)`**. **`value_portfolio`** returns **`PortfolioValuation`** JSON (see **`total_base_ccy`**). Helpers like **`portfolio_result_total_value`** expect the richer **`PortfolioResult`** envelope (**`valuation` + `metrics` + `meta`**); below we wrap the parsed valuation with minimal **metrics** and **meta** so the helper matches production reporting shapes.


In [3]:
from finstack.portfolio import portfolio_result_total_value, value_portfolio

portfolio = json.dumps(
    {
        "id": "credit-portfolio-2025",
        "as_of": "2025-01-15",
        "base_ccy": "USD",
        "entities": {"ACME": {"id": "ACME"}, "CASH": {"id": "CASH"}},
        "positions": [
            {
                "position_id": "POS-DEP",
                "entity_id": "CASH",
                "instrument_id": "DEP-6M",
                "instrument_spec": json.loads(deposit),
                "quantity": 1.0,
                "unit": "units",
            },
        ],
    }
)

val_json = value_portfolio(portfolio, market_json)
val_obj = json.loads(val_json)
pv_total = float(val_obj["total_base_ccy"]["amount"])
meta = {
    "numeric_mode": "F64",
    "rounding": {
        "mode": "Bankers",
        "ingest_scale_by_ccy": {},
        "output_scale_by_ccy": {},
        "version": 1,
        "tolerances": {"generic_epsilon": 1e-10, "rate_epsilon": 1e-12},
    },
}
metrics_payload = {
    "aggregated": {
        "pv": {
            "metric_id": "pv",
            "total": pv_total,
            "by_entity": {"CASH": pv_total},
        }
    },
    "by_position": {},
}
portfolio_result_json = json.dumps(
    {"valuation": val_obj, "metrics": metrics_payload, "meta": meta}
)
total = portfolio_result_total_value(portfolio_result_json)
print(f"Portfolio total (from valuation total_base_ccy): {pv_total:,.2f}")
print(f"portfolio_result_total_value (envelope): {total:,.2f}")
print("Valuation JSON top-level keys:", list(val_obj.keys()))


Portfolio total (from valuation total_base_ccy): -6,297.70
portfolio_result_total_value (envelope): -6,297.70
Valuation JSON top-level keys: ['as_of', 'position_values', 'total_base_ccy', 'by_entity']


## Step 4 — Issuer modeling

Build a minimal **ACME** P&L spec: explicit drivers (`revenue`, `cogs`, `opex`, `interest`) and computed nodes (`gross_profit`, `ebitda`, `ebt`). **`Evaluator`** produces a **`StatementResult`**; **`to_polars_wide()`** is convenient for scanning all line items across quarters.


In [4]:
from finstack.statements import Evaluator, ModelBuilder

b = ModelBuilder("acme-pl")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", [("2025Q1", 250.0), ("2025Q2", 260.0), ("2025Q3", 270.0), ("2025Q4", 280.0)])
b.value("cogs", [("2025Q1", 150.0), ("2025Q2", 155.0), ("2025Q3", 160.0), ("2025Q4", 165.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 50.0), ("2025Q2", 52.0), ("2025Q3", 54.0), ("2025Q4", 56.0)])
b.compute("ebitda", "gross_profit - opex")
b.value("interest", [("2025Q1", 10.0), ("2025Q2", 10.0), ("2025Q3", 10.0), ("2025Q4", 10.0)])
b.compute("ebt", "ebitda - interest")
spec = b.build()
result = Evaluator().evaluate(spec)
wide = result.to_polars_wide()
print(wide)
print(f"Wide frame shape: {wide.shape}")


shape: (4, 8)
┌───────────┬──────────┬──────┬───────┬─────────┬──────────────┬────────┬──────┐
│ period_id ┆ interest ┆ opex ┆ cogs  ┆ revenue ┆ gross_profit ┆ ebitda ┆ ebt  │
│ ---       ┆ ---      ┆ ---  ┆ ---   ┆ ---     ┆ ---          ┆ ---    ┆ ---  │
│ str       ┆ f64      ┆ f64  ┆ f64   ┆ f64     ┆ f64          ┆ f64    ┆ f64  │
╞═══════════╪══════════╪══════╪═══════╪═════════╪══════════════╪════════╪══════╡
│ 2025Q1    ┆ 10.0     ┆ 50.0 ┆ 150.0 ┆ 250.0   ┆ 100.0        ┆ 50.0   ┆ 40.0 │
│ 2025Q2    ┆ 10.0     ┆ 52.0 ┆ 155.0 ┆ 260.0   ┆ 105.0        ┆ 53.0   ┆ 43.0 │
│ 2025Q3    ┆ 10.0     ┆ 54.0 ┆ 160.0 ┆ 270.0   ┆ 110.0        ┆ 56.0   ┆ 46.0 │
│ 2025Q4    ┆ 10.0     ┆ 56.0 ┆ 165.0 ┆ 280.0   ┆ 115.0        ┆ 59.0   ┆ 49.0 │
└───────────┴──────────┴──────┴───────┴─────────┴──────────────┴────────┴──────┘
Wide frame shape: (4, 8)


## Step 5 — Stress testing

List **built-in scenario templates** and materialize one with **`build_from_template`** to show the JSON shape. Large composites (for example **`gfc_2008`**) assume **many** market objects (discount **and** CDS curves, equities, vol surfaces); a minimal self-contained capstone therefore applies a **custom** spec via **`build_scenario_spec`** (parallel **USD-OIS** shock) and runs **`apply_scenario_to_market`**. The returned dict includes **`operations_applied`** and stressed **`market_json`**.


In [5]:
from finstack.scenarios import (
    apply_scenario_to_market,
    build_from_template,
    build_scenario_spec,
    list_builtin_templates,
)

templates = list_builtin_templates()
print(f"Available templates ({len(templates)}): {templates}")

if templates:
    sample = build_from_template(templates[0])
    print(f"Sample template {templates[0]!r} JSON length: {len(sample):,} characters")
else:
    print("No built-in templates in this build.")

stress_ops = json.dumps(
    [
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "bp": 50.0,
        }
    ]
)
scenario_json = build_scenario_spec(
    "capstone-parallel-ois",
    stress_ops,
    name="Capstone +50bp OIS parallel",
)
stressed = apply_scenario_to_market(scenario_json, market_json, "2025-01-15")
print(f"Applied {stressed['operations_applied']} operations")
print(f"Stressed market_json length: {len(stressed['market_json']):,} characters")
if stressed.get("warnings"):
    print("Warnings:", stressed["warnings"])


Available templates (5): ['gfc_2008', 'covid_2020', 'rate_shock_2022', 'svb_2023', 'ltcm_1998']


ValueError: Market data not found: USD-SOFR

## Step 6 — Performance analysis

Simulate a **252-day** equity path with a fixed RNG seed, wrap it in **`Performance.from_arrays`**, and read **CAGR**, **Sharpe**, and **max drawdown** (per-series lists). Cross-check **annualized volatility**, **Sortino**, **VaR**, and **expected shortfall** using the standalone helpers on **simple returns**.


In [ ]:
import random
from datetime import timedelta

from finstack.analytics import (
    Performance,
    expected_shortfall,
    sharpe,
    simple_returns,
    sortino,
    value_at_risk,
    volatility,
)

random.seed(42)
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(252)]
prices = [100.0]
for _ in range(251):
    prices.append(prices[-1] * (1 + random.gauss(0.0003, 0.012)))

perf = Performance.from_arrays(dates, [prices], ["PORTFOLIO"])
print(f"CAGR: {perf.cagr()[0]:.6f}")
print(f"Sharpe (Performance): {perf.sharpe()[0]:.6f}")
print(f"Max DD: {perf.max_drawdown()[0]:.6f}")

r = simple_returns(prices)
ann_vol = volatility(r)
ann_mean = sum(r) / len(r) * 252.0
print(f"Ann. vol (standalone): {ann_vol:.6f}")
print(f"Sortino (standalone): {sortino(r):.6f}")
print(f"Sharpe from ann mean/vol: {sharpe(ann_mean, ann_vol):.6f}")
print(f"VaR (95%): {value_at_risk(r):.6f}")
print(f"ES (95%): {expected_shortfall(r):.6f}")


## Step 7 — Risk analytics

Estimate **factor beta** with **`calc_beta`** on aligned return samples, and **ruin probability** with **`estimate_ruin`** using a **wealth-floor** definition and a reproducible **`RuinModel`** seed.


In [ ]:
from finstack.analytics import RuinDefinition, RuinModel, calc_beta, estimate_ruin, simple_returns

rets = simple_returns(prices)
bench = [random.gauss(0.0004, 0.01) for _ in range(len(rets))]
beta_r = calc_beta(rets, bench)
print(f"Beta: {beta_r.beta:.4f}")

ruin = estimate_ruin(rets, RuinDefinition.wealth_floor(0.5), RuinModel(seed=42))
print(f"Ruin prob: {ruin.probability:.4f}")


## Step 8 — Reporting

Export the **model spec** and **evaluation result** to JSON, then use **`trace_dependencies`** for a dependency tree and **`explain_formula`** for a period-specific walkthrough of how **EBITDA** is computed.


In [ ]:
from finstack.statements_analytics import explain_formula, trace_dependencies

model_json = spec.to_json()
eval_json = result.to_json()
deps = trace_dependencies(model_json, "ebitda")
print("EBITDA dependencies:")
print(deps)

explanation = explain_formula(model_json, eval_json, "ebitda", "2025Q1")
print("Formula explanation:")
print(explanation)


## Summary

You walked the full stack:

- **Core market data** as JSON shared by pricers and the portfolio engine.
- **Valuations** and **portfolio** aggregation over the same instrument spec.
- **Statements** for issuer-level P&L with **analytics** for dependency tracing and formula explanation.
- **Scenarios** for template-driven market stress.
- **Analytics** for performance, tail risk, beta, and ruin-style diagnostics.

From here, extend the portfolio with bonds or loans, swap in calibrated curves, attach entity-level statement models per obligor, and wire scenario results into your own limits and reporting stack — still using the same JSON-first contracts throughout.
